# 02 — Forecasting backtest

Compare naive / seasonal-naive / Ridge / LightGBM with an expanding-window backtest (no leakage).

In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
from demand_forecaster.data import load_demand
from demand_forecaster.backtest import run_backtest_suite

_, y = load_demand(series='item_1')
results = run_backtest_suite(y, horizon=28, min_train=365, step=28)
lb = pd.DataFrame([{**v['overall'], 'model': k} for k, v in results.items()]).sort_values('mae')
lb

In [ ]:
from copy import deepcopy
from demand_forecaster.models import default_model_zoo
from demand_forecaster.backtest import holdout_eval
import matplotlib.pyplot as plt

model = deepcopy(default_model_zoo()['lightgbm'])
out = holdout_eval(y, model, test_size=90)
print(out['metrics'])
plt.figure(figsize=(11, 4))
plt.plot(out['test'].index, out['test'].values, label='actual')
plt.plot(out['pred'].index, out['pred'].values, label='forecast', linestyle='--')
plt.legend(); plt.title('LightGBM holdout'); plt.show()
if out['model'].feature_importances_:
    pd.Series(out['model'].feature_importances_).head(12).plot(kind='barh', title='Drivers')
    plt.show()